# **Phase 5 — Marketing Insights Dashboard**

**Goal:** -> Architect and build an interactive enterprise-grade marketing dashboard using **Plotly and Dash** that unifies analytics and insights from all previous phases (Data Preprocessing, RFM Analysis, K-Means Clustering, and Customer Lifetime Value Forecasting).

This system bridges technical modeling and strategic decision-making, allowing marketing professionals to filter customer databases in real-time, inspect automated high-value risk flags, and export custom campaign lists.

---
### **Key Modules Unified inside this System:**
1. **Behavioral Layer:** Core transactional properties (`region`, `category`, `net_revenue`) from **Phase 1**.
2. **Segmentation Layer:** Micro-segments based on Recency and Frequency from **Phase 2**.
3. **Clustering Layer:** Natural multi-dimensional clusters generated via K-Means in **Phase 3**.
4. **Predictive Layer:** Future purchase probabilities, 365-day CLV forecasts, and automated risk markers via the **BG/NBD and Gamma-Gamma** framework in **Phase 4**.


In [3]:
import os
import pandas as pd
import numpy as np
import warnings
import plotly # Imported the base module for the version check
import plotly.express as px
import plotly.graph_objects as go
import dash
from dash import dcc, html, dash_table, Input, Output, State

# Suppress warnings
warnings.filterwarnings('ignore')

# Pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✓ Libraries loaded successfully for Dashboard Development")
print(f"  - Dash version: {dash.__version__}")
print(f"  - Plotly version: {plotly.__version__}") # Fixed this line

✓ Libraries loaded successfully for Dashboard Development
  - Dash version: 4.2.0
  - Plotly version: 6.6.0


## **Section 1: Data Loading & Enterprise Integration**

To construct a comprehensive dashboard ecosystem, we combine two major levels of data:
1. **Transactional Granularity (`01_preprocessed.csv`):** Houses item-level detail, categories, sales regions, dates, and calculated margins.
2. **Customer Analytical Profile (`04_clv.csv`):** Houses aggregated behavioral characteristics, algorithmic segmentation labels, and future monetary metrics.

By joining these datasets via `customer_id`, we establish an analytical scope capable of slicing high-level metrics by localized product and demographic features.


In [4]:
# Define explicit paths to upstream assets
tx_path = './data/01_preprocessed.csv'
clv_path = './data/04_clv.csv'

# Validate file presence before running pipeline
if not os.path.exists(tx_path) or not os.path.exists(clv_path):
    raise FileNotFoundError(
        "Required upstream data dependencies missing. Please execute Phase 1 and Phase 4 notebooks first."
    )

# Read assets into memory
df_tx = pd.read_csv(tx_path)
df_clv = pd.read_csv(clv_path)

# Ensure proper time series format
df_tx['order_date'] = pd.to_datetime(df_tx['order_date'])

# Isolate unique predictive features to avoid duplicate multi-join overlapping suffixes
clv_features = ['customer_id', 'pred_365d', 'CLV_365d', 'CLV_Tier', 'at_risk_flag', 'Segment', 'RFM_Score']
df_clv_clean = df_clv[clv_features]

# Execute left join to generate Unified Master Frame
df_master = pd.merge(df_tx, df_clv_clean, on='customer_id', how='left')

print("=== DATA INTEGRATION SUMMARY ===")
print(f"✓ Master Transactional Records Joined : {df_master.shape[0]:,} rows × {df_master.shape[1]} columns")
print(f"✓ Unique Customer Profiles Integrated : {df_clv.shape[0]:,} individual entities")
print(f"✓ Active Temporal Horizon            : {df_master['order_date'].min().strftime('%Y-%m-%d')} → {df_master['order_date'].max().strftime('%Y-%m-%d')}")


=== DATA INTEGRATION SUMMARY ===
✓ Master Transactional Records Joined : 34,500 rows × 30 columns
✓ Unique Customer Profiles Integrated : 7,903 individual entities
✓ Active Temporal Horizon            : 2023-09-12 → 2025-09-11


## **Section 2: Dashboard Custom Layout & Aesthetics Configuration**

We apply a cohesive professional palette using soft, desaturated accent colors. This avoids harsh, primary web colors to emphasize the quantitative data charts. 

We also extract structural dimensions (unique regions, product types, clusters) to feed the inputs of the interactive global filtering matrix.


In [5]:
# Professional desaturated executive styling theme
THEME = {
    'bg_canvas': '#F8FAFC',     # Slate background
    'bg_card': '#FFFFFF',       # Crisp white cards
    'border': '#E2E8F0',        # Light boundary grey
    'text_main': '#0F172A',     # Deep obsidian text
    'text_muted': '#64748B',    # Muted slate text
    'primary': '#4F46E5',       # Indigo primary indicator
    'secondary': '#0EA5E9',     # Calm sky blue
    'accent': '#F59E0B',        # Balanced gold/amber
    'danger': '#EF4444'         # Clean validation red
}

# Extract unique dimensions for selection drop-downs
regions_list = sorted(df_master['region'].dropna().unique())
categories_list = sorted(df_master['category'].dropna().unique())
segments_list = sorted(df_master['Segment'].dropna().unique())
clv_tiers_list = ['Platinum', 'Gold', 'Silver', 'Bronze']

print("✓ Visual configuration environment built.")
print(f"  - Extracted Regions    : {regions_list}")
print(f"  - Extracted Categories : {categories_list}")
print(f"  - Extracted Segments   : {segments_list}")


✓ Visual configuration environment built.
  - Extracted Regions    : ['Central', 'East', 'North', 'South', 'West']
  - Extracted Categories : ['Beauty', 'Electronics', 'Fashion', 'Grocery', 'Home', 'Sports', 'Toys']
  - Extracted Segments   : ['At Risk', "Can't Lose Them", 'Champions', 'Hibernating', 'Loyal Customers', 'Needs Attention', 'Potential Loyalists']


## **Section 3: Interface Layout Assembly**

Using Dash Core (`dcc`) and HTML (`html`) elements, we construct a responsive visual workspace split into three logical functional zones:
1. **Global Control Sidebar:** Holds interactive dropdown select menus that sync changes globally across all performance plots instantly.
2. **Executive Level Metrics Row:** Displays real-time summary indicators (Net Revenue, Customer Count, Mean CLV, Churn Hazards).
3. **Deep-Dive Graphic Canvas:** Combines a scatter value matrix, value vs. volume distribution bars, regional performance pies, category affinity graphs, and an exportable multi-page data matrix.


In [6]:
# Initialize App Wrapper
app = dash.Dash(__name__, title="Enterprise Customer Analytics Hub")

# Construct Comprehensive Component Trees
app.layout = html.Div(
    style={'backgroundColor': THEME['bg_canvas'], 'fontFamily': 'Segoe UI, Helvetica, sans-serif', 'color': THEME['text_main'], 'padding': '24px', 'margin': '0'},
    children=[
        
        # Dashboard Top Branding Strip
        html.Div(
            style={'backgroundColor': THEME['bg_card'], 'padding': '24px 32px', 'borderRadius': '12px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)', 'marginBottom': '24px', 'display': 'flex', 'justifyContent': 'space-between', 'alignItems': 'center', 'borderLeft': f'6px solid {THEME["primary"]}'},
            children=[
                html.Div([
                    html.H1("E-Commerce Customer Segmentation & CLV Hub", style={'margin': '0 0 4px 0', 'fontSize': '26px', 'fontWeight': '700'}),
                    html.P("Phase 5 — Interactive Marketing Insights Engine & Campaign Pipeline", style={'margin': '0', 'color': THEME['text_muted'], 'fontSize': '14px'})
                ]),
                html.Div(
                    style={'backgroundColor': '#F1F5F9', 'padding': '8px 16px', 'borderRadius': '8px', 'fontWeight': '600', 'fontSize': '13px'},
                    children=f"Engine State: Ready"
                )
            ]
        ),
        
        # Primary Content Layout Workspace Grid
        html.Div(
            style={'display': 'table', 'width': '100%', 'borderSpacing': '20px 0px', 'margin': '0 -20px'},
            children=[
                
                # LEFT FILTER COLUMN PANEL
                html.Div(
                    style={'display': 'table-cell', 'width': '280px', 'verticalAlign': 'top', 'backgroundColor': THEME['bg_card'], 'padding': '24px', 'borderRadius': '12px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'},
                    children=[
                        html.H3("Global Search Criteria", style={'margin': '0 0 20px 0', 'fontSize': '16px', 'fontWeight': '700', 'borderBottom': f'1px solid {THEME["border"]}', 'paddingBottom': '10px'}),
                        
                        html.Div([
                            html.Label("Sales Region", style={'fontWeight': '600', 'fontSize': '12px', 'color': THEME['text_muted'], 'display': 'block', 'marginBottom': '6px'}),
                            dcc.Dropdown(id='region-filter', options=[{'label': r, 'value': r} for r in regions_list], multi=True, placeholder="All Regions", style={'marginBottom': '20px'})
                        ]),
                        
                        html.Div([
                            html.Label("Product Category", style={'fontWeight': '600', 'fontSize': '12px', 'color': THEME['text_muted'], 'display': 'block', 'marginBottom': '6px'}),
                            dcc.Dropdown(id='category-filter', options=[{'label': c, 'value': c} for c in categories_list], multi=True, placeholder="All Categories", style={'marginBottom': '20px'})
                        ]),
                        
                        html.Div([
                            html.Label("RFM Customer Segment", style={'fontWeight': '600', 'fontSize': '12px', 'color': THEME['text_muted'], 'display': 'block', 'marginBottom': '6px'}),
                            dcc.Dropdown(id='segment-filter', options=[{'label': s, 'value': s} for s in segments_list], multi=True, placeholder="All Segments", style={'marginBottom': '20px'})
                        ]),
                        
                        html.Div([
                            html.Label("Predictive CLV Tier", style={'fontWeight': '600', 'fontSize': '12px', 'color': THEME['text_muted'], 'display': 'block', 'marginBottom': '6px'}),
                            dcc.Dropdown(id='clv-filter', options=[{'label': t, 'value': t} for t in clv_tiers_list], multi=True, placeholder="All Tiers", style={'marginBottom': '24px'})
                        ]),
                        
                        html.Button(
                            "Clear Filter Matrix", id='reset-btn', n_clicks=0,
                            style={'width': '100%', 'padding': '10px', 'backgroundColor': '#F1F5F9', 'color': THEME['text_main'], 'border': 'none', 'borderRadius': '6px', 'fontWeight': '600', 'cursor': 'pointer'}
                        )
                    ]
                ),
                
                # RIGHT VISUALIZATION HUB CANVAS
                html.Div(
                    style={'display': 'table-cell', 'verticalAlign': 'top'},
                    children=[
                        
                        # EXECUTION KPI ROW INDICATORS
                        html.Div(
                            style={'display': 'flex', 'gap': '16px', 'marginBottom': '24px'},
                            children=[
                                html.Div(id='kpi-rev', style={'flex': '1', 'backgroundColor': THEME['bg_card'], 'padding': '20px', 'borderRadius': '12px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)', 'borderTop': f'4px solid {THEME["primary"]}'}),
                                html.Div(id='kpi-cust', style={'flex': '1', 'backgroundColor': THEME['bg_card'], 'padding': '20px', 'borderRadius': '12px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)', 'borderTop': f'4px solid {THEME["secondary"]}'}),
                                html.Div(id='kpi-clv', style={'flex': '1', 'backgroundColor': THEME['bg_card'], 'padding': '20px', 'borderRadius': '12px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)', 'borderTop': f'4px solid {THEME["accent"]}'}),
                                html.Div(id='kpi-risk', style={'flex': '1', 'backgroundColor': THEME['bg_card'], 'padding': '20px', 'borderRadius': '12px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)', 'borderTop': f'4px solid {THEME["danger"]}'})
                            ]
                        ),
                        
                        # ROW 1 GRAPH PANELS
                        html.Div(
                            style={'display': 'flex', 'gap': '20px', 'marginBottom': '24px'},
                            children=[
                                html.Div(
                                    style={'flex': '1', 'backgroundColor': THEME['bg_card'], 'padding': '20px', 'borderRadius': '12px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)', 'width': '50%'},
                                    children=[
                                        html.H4("RFM Value Mapping (Recency vs Monetary Cluster Matrix)", style={'margin': '0 0 12px 0', 'fontSize': '14px', 'fontWeight': '600', 'color': THEME['text_muted']}),
                                        dcc.Graph(id='scatter-rfm', config={'displayModeBar': False})
                                    ]
                                ),
                                html.Div(
                                    style={'flex': '1', 'backgroundColor': THEME['bg_card'], 'padding': '20px', 'borderRadius': '12px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)', 'width': '50%'},
                                    children=[
                                        html.H4("Segment Population Share vs Projected CLV Share", style={'margin': '0 0 12px 0', 'fontSize': '14px', 'fontWeight': '600', 'color': THEME['text_muted']}),
                                        dcc.Graph(id='bar-segmentation', config={'displayModeBar': False})
                                    ]
                                )
                            ]
                        ),
                        
                        # ROW 2 GRAPH PANELS
                        html.Div(
                            style={'display': 'flex', 'gap': '20px', 'marginBottom': '24px'},
                            children=[
                                html.Div(
                                    style={'flex': '1', 'backgroundColor': THEME['bg_card'], 'padding': '20px', 'borderRadius': '12px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)', 'width': '50%'},
                                    children=[
                                        html.H4("Product Categories Ranked by Aggregated Contribution", style={'margin': '0 0 12px 0', 'fontSize': '14px', 'fontWeight': '600', 'color': THEME['text_muted']}),
                                        dcc.Graph(id='bar-categories', config={'displayModeBar': False})
                                    ]
                                ),
                                html.Div(
                                    style={'flex': '1', 'backgroundColor': THEME['bg_card'], 'padding': '20px', 'borderRadius': '12px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)', 'width': '50%'},
                                    children=[
                                        html.H4("Geographic Location Value Concentration", style={'margin': '0 0 12px 0', 'fontSize': '14px', 'fontWeight': '600', 'color': THEME['text_muted']}),
                                        dcc.Graph(id='pie-regions', config={'displayModeBar': False})
                                    ]
                                )
                            ]
                        ),
                        
                        # ROW 3 DATA ENGINE EXPORT GENERATOR GRID
                        html.Div(
                            style={'backgroundColor': THEME['bg_card'], 'padding': '24px', 'borderRadius': '12px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)', 'marginBottom': '24px'},
                            children=[
                                html.Div(
                                    style={'display': 'flex', 'justifyContent': 'space-between', 'alignItems': 'center', 'marginBottom': '16px'},
                                    children=[
                                        html.Div([
                                            html.H4("Targeted Campaign List Generator", style={'margin': '0 0 4px 0', 'fontSize': '16px', 'fontWeight': '700'}),
                                            html.P("Synchronized dynamically by choices above. Ready for multi-channel injection.", style={'margin': '0', 'color': THEME['text_muted'], 'fontSize': '12px'})
                                        ]),
                                        html.Button(
                                            "Export Action List (.CSV)", id="btn-csv-export",
                                            style={'padding': '10px 16px', 'backgroundColor': THEME['primary'], 'color': '#FFFFFF', 'border': 'none', 'borderRadius': '6px', 'fontWeight': '600', 'cursor': 'pointer'}
                                        ),
                                        dcc.Download(id="download-engine")
                                    ]
                                ),
                                dash_table.DataTable(
                                    id='campaign-data-grid',
                                    columns=[
                                        {"name": "Customer ID", "id": "customer_id"},
                                        {"name": "RFM Segment", "id": "Segment"},
                                        {"name": "Composite Score", "id": "RFM_Score"},
                                        {"name": "CLV Tier", "id": "CLV_Tier"},
                                        {"name": "Exp. Orders (365d)", "id": "pred_365d"},
                                        {"name": "Forecasted CLV (₹)", "id": "CLV_365d"},
                                        {"name": "At-Risk Status", "id": "at_risk_flag"}
                                    ],
                                    page_size=8,
                                    style_table={'overflowX': 'auto'},
                                    style_header={'backgroundColor': '#F8FAFC', 'fontWeight': 'bold', 'color': THEME['text_main'], 'border': f'1px solid {THEME["border"]}', 'padding': '10px'},
                                    style_cell={'padding': '10px', 'fontSize': '12px', 'textAlign': 'left', 'borderBottom': f'1px solid THEME["border"]'},
                                    style_data_conditional=[
                                        {'if': {'row_index': 'odd'}, 'backgroundColor': '#FAFAFA'},
                                        {'if': {'column_id': 'at_risk_flag', 'filter_query': '{at_risk_flag} eq 1'}, 'color': '#EF4444', 'fontWeight': '700', 'backgroundColor': '#FEF2F2'}
                                    ]
                                )
                            ]
                        )
                        
                    ]
                )
            ]
        )
    ]
)
print("✓ Application Layout Blueprint instantiated successfully.")


✓ Application Layout Blueprint instantiated successfully.


## **Section 4: Server Callback Pipeline & Reactive Computations**

Now, we implement the interactive logic mapping inputs to outputs. Every time a dropdown component changes, Dash automatically feeds those values into this pipeline, dynamically recalculates the subsets of rows from the dataframes, rebuilds the corresponding figures, and populates the KPI metrics.


In [7]:
# Callback to handle dropdown value clearing via the reset button
@app.callback(
    [Output('region-filter', 'value'),
     Output('category-filter', 'value'),
     Output('segment-filter', 'value'),
     Output('clv-filter', 'value')],
    [Input('reset-btn', 'n_clicks')]
)
def handle_filter_flush(n_clicks):
    return None, None, None, None


# Master Data Sync Pipeline Callback
@app.callback(
    [Output('kpi-rev', 'children'),
     Output('kpi-cust', 'children'),
     Output('kpi-clv', 'children'),
     Output('kpi-risk', 'children'),
     Output('scatter-rfm', 'figure'),
     Output('bar-segmentation', 'figure'),
     Output('bar-categories', 'figure'),
     Output('pie-regions', 'figure'),
     Output('campaign-data-grid', 'data')],
    [Input('region-filter', 'value'),
     Input('category-filter', 'value'),
     Input('segment-filter', 'value'),
     Input('clv-filter', 'value')]
)
def filter_and_render_canvas(regions_sel, categories_sel, segments_sel, tiers_sel):
    
    # Isolate records based on raw transactional filters
    tx_sub = df_master.copy()
    if regions_sel:
        tx_sub = tx_sub[tx_sub['region'].isin(regions_sel)]
    if categories_sel:
        tx_sub = tx_sub[tx_sub['category'].isin(categories_sel)]
    if segments_sel:
        tx_sub = tx_sub[tx_sub['Segment'].isin(segments_sel)]
    if tiers_sel:
        tx_sub = tx_sub[tx_sub['CLV_Tier'].isin(tiers_sel)]
        
    # Match structural transactional results back to unique customer level entities
    active_ids = tx_sub['customer_id'].unique()
    cust_sub = df_clv[df_clv['customer_id'].isin(active_ids)].copy()
    
    # 1. Recalculate KPIs
    total_rev = tx_sub['net_revenue'].sum()
    total_custs = len(cust_sub)
    mean_clv = cust_sub['CLV_365d'].mean() if total_custs > 0 else 0
    risk_hazards = cust_sub['at_risk_flag'].sum() if total_custs > 0 else 0
    
    kpi_rev_el = [html.H5("Calculated Net Sales", style={'margin': '0 0 4px 0', 'color': THEME['text_muted'], 'fontSize': '11px', 'textTransform': 'uppercase'}), html.H2(f"₹{total_rev:,.2f}", style={'margin': '0', 'fontSize': '20px', 'fontWeight': '700', 'color': THEME['primary']})]
    kpi_cust_el = [html.H5("Filtered Cohort Size", style={'margin': '0 0 4px 0', 'color': THEME['text_muted'], 'fontSize': '11px', 'textTransform': 'uppercase'}), html.H2(f"{total_custs:,} Users", style={'margin': '0', 'fontSize': '20px', 'fontWeight': '700', 'color': THEME['secondary']})]
    kpi_clv_el = [html.H5("Mean Expected CLV", style={'margin': '0 0 4px 0', 'color': THEME['text_muted'], 'fontSize': '11px', 'textTransform': 'uppercase'}), html.H2(f"₹{mean_clv:,.2f}", style={'margin': '0', 'fontSize': '20px', 'fontWeight': '700', 'color': THEME['accent']})]
    kpi_risk_el = [html.H5("High Value At Risk", style={'margin': '0 0 4px 0', 'color': THEME['text_muted'], 'fontSize': '11px', 'textTransform': 'uppercase'}), html.H2(f"{int(risk_hazards):,} Profiles", style={'margin': '0', 'fontSize': '20px', 'fontWeight': '700', 'color': THEME['danger']})]
    
    # 2. Render Figure 1: Scatter RFM Matrix
    if not cust_sub.empty:
        # Prevent graph skew by capping extreme monetary outliers at 98th percentile for visualization space
        m_ceiling = cust_sub['monetary_value'].quantile(0.98) if 'monetary_value' in cust_sub.columns else 10000
        scatter_df = cust_sub.copy()
        y_col = 'monetary_value' if 'monetary_value' in scatter_df.columns else 'CLV_365d'
        scatter_df['Monetary_Visual'] = scatter_df[y_col].clip(upper=m_ceiling)
        
        fig_scatter = px.scatter(
            scatter_df, x='Recency', y='Monetary_Visual', color='Segment', size='CLV_365d',
            hover_data=['customer_id', 'RFM_Score', 'CLV_Tier'],
            color_discrete_sequence=px.colors.qualitative.Safe
        )
        fig_scatter.update_layout(
            margin={'l': 10, 'r': 10, 't': 10, 'b': 10}, height=280,
            paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
          xaxis={'title': {'text': 'Recency (Days Inactive)', 'font': {'size': 11}}, 'gridcolor': THEME['border']},
          yaxis={'title': {'text': 'Value Metric (₹ Capped)', 'font': {'size': 11}}, 'gridcolor': THEME['border']},
            showlegend=False # <-- This is the corrected line!
         )
    else:
        fig_scatter = go.Figure()

    # 3. Render Figure 2: Volume Share vs Revenue Contribution
    if not cust_sub.empty:
        v_share = cust_sub['Segment'].value_counts(normalize=True).reset_index(name='Volume %')
        r_share = cust_sub.groupby('Segment')['CLV_365d'].sum().reset_index()
        r_sum = r_share['CLV_365d'].sum()
        r_share['Revenue %'] = r_share['CLV_365d'] / (r_sum if r_sum > 0 else 1)
        
        merged_share = pd.merge(v_share, r_share, on='Segment')
        melted = merged_share.melt(id_vars='Segment', value_vars=['Volume %', 'Revenue %'], var_name='Metric', value_name='Share')
        melted['Share'] *= 100
        
        fig_bar_share = px.bar(
            melted, x='Segment', y='Share', color='Metric', barmode='group',
            color_discrete_sequence=[THEME['secondary'], THEME['primary']]
        )
        fig_bar_share.update_layout(
            margin={'l': 10, 'r': 10, 't': 10, 'b': 10}, height=280,
            paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
            xaxis={'title': '', 'tickangle': 20, 'tickfont': {'size': 10}},
            yaxis={'title': {'text': 'Percentage Share (%)', 'font': {'size': 11}}, 'gridcolor': THEME['border']},
            legend={'title': '', 'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'xanchor': 'right', 'x': 1}
        )
    else:
        fig_bar_share = go.Figure()

    # 4. Render Figure 3: Category Contribution Rank
    if not tx_sub.empty:
        cat_data = tx_sub.groupby('category')['net_revenue'].sum().reset_index().sort_values('net_revenue', ascending=True)
        fig_cat = px.bar(cat_data, x='net_revenue', y='category', orientation='h', color_discrete_sequence=[THEME['primary']])
        fig_cat.update_layout(
            margin={'l': 10, 'r': 10, 't': 10, 'b': 10}, height=260,
            paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
            xaxis={'title': {'text': 'Calculated Value (₹)', 'font': {'size': 11}}, 'gridcolor': THEME['border']},
            yaxis={'title': ''}
        )
    else:
        fig_cat = go.Figure()

    # 5. Render Figure 4: Regional Distribution Pie
    if not tx_sub.empty:
        reg_data = tx_sub.groupby('region')['net_revenue'].sum().reset_index()
        fig_pie = px.pie(reg_data, values='net_revenue', names='region', color_discrete_sequence=px.colors.sequential.Teal_r)
        fig_pie.update_layout(margin={'l': 10, 'r': 10, 't': 10, 'b': 10}, height=260, paper_bgcolor='rgba(0,0,0,0)')
        fig_pie.update_traces(textposition='inside', textinfo='percent+label', textfont={'size': 10})
    else:
        fig_pie = go.Figure()

    # 6. Parse Data Rows for Data Table Grid View
    grid_records = cust_sub.round(2).to_dict('records')
    
    return kpi_rev_el, kpi_cust_el, kpi_clv_el, kpi_risk_el, fig_scatter, fig_bar_share, fig_cat, fig_pie, grid_records


# Callback to manage targeted segment data extractions and file downloads
@app.callback(
    Output("download-engine", "data"),
    Input("btn-csv-export", "n_clicks"),
    [State('region-filter', 'value'),
     State('category-filter', 'value'),
     State('segment-filter', 'value'),
     State('clv-filter', 'value')],
    prevent_initial_call=True
)
def manage_file_export_download(n_clicks, r_f, c_f, s_f, t_f):
    tx_export = df_master.copy()
    if r_f: tx_export = tx_export[tx_export['region'].isin(r_f)]
    if c_f: tx_export = tx_export[tx_export['category'].isin(c_f)]
    if s_f: tx_export = tx_export[tx_export['Segment'].isin(s_f)]
    if t_f: tx_export = tx_export[tx_export['CLV_Tier'].isin(t_f)]
        
    matched_export_ids = tx_export['customer_id'].unique()
    cust_export = df_clv[df_clv['customer_id'].isin(matched_export_ids)]
    
    return dcc.send_data_frame(cust_export.to_csv, "targeted_marketing_list.csv", index=False)

print("✓ Interactive backend callback functions successfully compiled.")


✓ Interactive backend callback functions successfully compiled.


## **Section 5: Activating the Dashboard Server Environment**

Dash natively supports direct interactive cell execution within Jupyter Notebooks. 
- Setting `jupyter_mode='inline'` renders the dashboard application directly below this cell.
- Setting `jupyter_mode='external'` registers a local background server and provides an active URL link (e.g., `http://127.0.0.1:8050/`) to view the application in a full browser tab.

*Run the cell below to launch the live analytics dashboard application:*


In [8]:
if __name__ == '__main__':
    # Switch to jupyter_mode='inline' if you wish to render it inside the notebook canvas directly
    print("Launching Local Dash Web App...")
    app.run(jupyter_mode='external', port=8050, debug=True)


Launching Local Dash Web App...
Dash app running on http://127.0.0.1:8050/
